In [3]:
import os
import json
import base64
from typing import List

from unstructured.staging.base import elements_to_json, elements_from_json
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output

from io import StringIO
import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

load_dotenv()

os.makedirs("extracted_data/images", exist_ok=True)
os.makedirs("extracted_data/tables", exist_ok=True)

In [4]:
def partition_document(file_path: str, cache_path: str = None):
    """Extract elements from PDF, using a cached JSON if available"""

    if cache_path is None:
        cache_path = file_path.replace(".pdf", "_elements.json")

    if os.path.exists(cache_path):
        print(f"✅ Found cached elements at {cache_path} — loading instead of re-parsing")
        elements = elements_from_json(cache_path)
        print(f"✅ Loaded {len(elements)} elements from cache")
        return elements

    print(f"📄 Partitioning document: {file_path} (this may take a while for hi_res)")

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )

    print(f"✅ Extracted {len(elements)} elements")

    elements_to_json(elements, filename=cache_path)
    print(f"💾 Cached elements to {cache_path} for next run")

    return elements


file_path = "docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

✅ Found cached elements at docs/attention-is-all-you-need_elements.json — loading instead of re-parsing
✅ Loaded 266 elements from cache


In [6]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")

    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )

    print(f"✅ Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 33 chunks


In [26]:
chunks[0].to_dict()

{'type': 'CompositeElement',
 'element_id': '8b1deff5-e152-4529-8f8c-6585db0ff0ee',
 'text': 'Design and build accessible PDF tables Sample tables\n\nTable 1',
 'metadata': {'file_directory': 'docs',
  'filename': 'sample-tables.pdf',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'last_modified': '2026-07-29T12:36:32',
  'page_number': 1,
  'orig_elements': 'eJzdUsFu4yAU/BXEOfFijMHkXO25UnOLIgvDs4OEsWWwtlHVfy/YyaXqaaW97HHmzZj3xnP5wOBgBB9ba/AJYSM7Dow2tWBcAOG95gyILHtKtBa6wQeER4jKqKiS/gPraVqM9SpC2LBT92mN7Q3scIuJqZqaJM+D/mNNvCWWSlEndp6sj9l3ubCKFeyAGK2K5npATyw4KWTGJRe8qH4gdkdicLiHCGO+4tW+g3ublQb8mQYGIuhoJ99qp0Jo52XqkowUXNKqSYLeOmiNXZJqWu5bDpMO+DHxaoTMBTXODo5RdQ5CMZv+KYj3eROoeXZWq/zSr8fYhhbe46J0hC3fuKyAtzz8sKphC+2CwQ/4urEhtuNkbG93OSWUH4k4Unku6anip4pm95ycrV/HDpakKvONMT2THS8Q7OCR8gZ1q3UGKa0hBJt2Rq8vv9G+PXrbbnmg/MnnDWcb3Zba92ZIIyhnTAvJoCaS1ZRT05dM9VwTI6p/2Ayem9BIWfC9GRuWNS/qjLmocw++413/d72oS06r/6sX57wdKn/41dcvE5g2hQ=='}}

In [7]:
plt.rcParams['font.family'] = 'DejaVu Sans'  # better unicode/math glyph coverage


def crop_table_from_pdf(pdf_path, page_number, element, output_path, zoom=3, padding=8):
    """Crop the exact table region from the original PDF page — pixel-perfect, no reconstruction"""
    try:
        coords = element.metadata.coordinates
        if not coords or not coords.points:
            return None

        xs = [p[0] for p in coords.points]
        ys = [p[1] for p in coords.points]
        x0, x1 = min(xs) - padding, max(xs) + padding
        y0, y1 = min(ys) - padding, max(ys) + padding

        doc = fitz.open(pdf_path)
        page = doc[page_number - 1]  # unstructured page_number is 1-indexed

        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, clip=fitz.Rect(x0, y0, x1, y1))
        pix.save(output_path)
        doc.close()
        return output_path

    except Exception as e:
        print(f"     ⚠️ Could not crop table from PDF: {e}")
        return None


def clean_cell_text(cell):
    """Extract cell text, converting <sup>/<sub> tags to unicode where possible"""
    sup_map = {'2': '²', '3': '³', '1': '¹'}
    text = ''
    for content in cell.contents:
        if getattr(content, 'name', None) == 'sup':
            raw = content.get_text()
            text += sup_map.get(raw, f'^{raw}')
        elif getattr(content, 'name', None) == 'sub':
            text += f"_{content.get_text()}"
        else:
            text += str(content) if isinstance(content, str) else content.get_text()
    return text.strip()


def html_table_to_image(html, output_path, title=None):
    """Render an HTML table as a clean PNG, preserving super/subscripts"""
    try:
        soup = BeautifulSoup(html, 'html.parser')
        rows = soup.find_all('tr')
        if not rows:
            print(f"     ⚠️ No rows found in table HTML — skipping image render")
            return None

        data = [[clean_cell_text(c) for c in row.find_all(['td', 'th'])] for row in rows]
        header, body = data[0], data[1:]

        # NEW: guard against tables with no body rows, or malformed/ragged rows
        if not body or not header:
            print(f"     ⚠️ Table has no usable data rows — skipping image render")
            return None

        # NEW: pad/truncate rows to match header length (handles merged/spanning cells)
        col_count = len(header)
        body = [row + [''] * (col_count - len(row)) if len(row) < col_count else row[:col_count] for row in body]

        df = pd.DataFrame(body, columns=header)

    except Exception as e:
        print(f"     ⚠️ Could not parse table HTML: {e}")
        return None

    if df.empty:
        print(f"     ⚠️ Parsed table is empty — skipping image render")
        return None

    fig, ax = plt.subplots(figsize=(max(6, len(df.columns) * 1.4), max(1.5, len(df) * 0.5 + 1)))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold', pad=12)

    tbl = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#4a4a4a')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#f5f5f5')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    return output_path

In [8]:
def separate_content_types(chunk, chunk_id):
    """Analyze content types AND persist images/tables to disk with paths"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text'],
        'page': None
    }

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if content_data['page'] is None and hasattr(element.metadata, 'page_number'):
                content_data['page'] = element.metadata.page_number

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)

                table_idx = len(content_data['tables'])
                table_path = f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html"
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{table_html}</body></html>")

                img_path = f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png"
                rendered = html_table_to_image(table_html, img_path, title=f"Table (page {content_data.get('page')})")

                content_data['tables'].append({"html": table_html, "path": table_path, "image_path": rendered})

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    img_b64 = element.metadata.image_base64

                    img_idx = len(content_data['images'])
                    img_path = f"extracted_data/images/chunk_{chunk_id}_img_{img_idx}.png"
                    with open(img_path, 'wb') as f:
                        f.write(base64.b64decode(img_b64))

                    content_data['images'].append({"base64": img_b64, "path": img_path})

    content_data['types'] = list(set(content_data['types']))
    return content_data

In [9]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    try:
        model_name = "qwen/qwen3.6-27b" if images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}
        """

        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description covering key facts, main topics,
        questions this content could answer, visual content analysis, and alternative search terms.

        SEARCHABLE DESCRIPTION:"""

        if images:
            message_content = [{"type": "text", "text": prompt_text}]
            for image_base64 in images:
                message_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                })
        else:
            message_content = prompt_text

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [10]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries, saving images/tables to disk"""
    print("🧠 Processing chunks with AI Summaries...")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        print(f"   Processing chunk {i+1}/{total_chunks}")

        content_data = separate_content_types(chunk, chunk_id=i)
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        table_htmls = [t["html"] for t in content_data['tables']]
        image_b64s = [img["base64"] for img in content_data['images']]

        if table_htmls or image_b64s:
            print("     → Creating AI summary for mixed content...")
            enhanced_content = create_ai_enhanced_summary(content_data['text'], table_htmls, image_b64s)
        else:
            print("     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        doc = Document(
            page_content=enhanced_content,
            metadata={
                "chunk_id": i,
                "page": content_data['page'] or 0,
                "raw_text": content_data['text'][:2000],
                "table_paths": json.dumps([t["path"] for t in content_data['tables']]),
                "table_image_paths": json.dumps([t["image_path"] for t in content_data['tables'] if t.get("image_path")]),
                "image_paths": json.dumps([img["path"] for img in content_data['images']]),
                "has_table": bool(content_data['tables']),
                "has_image": bool(content_data['images']),
            }
        )
        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/33
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
   Processing chunk 6/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 7/33
     Types found: ['image', 'text']
     Tables: 0, Images: 2
     → Creating AI summary for mixed content...
   Processing chunk 8/33
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw

KeyboardInterrupt: 

In [36]:
class NomicEmbeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        prefixed = [f"search_document: {t}" for t in texts]
        return super().embed_documents(prefixed)

    def embed_query(self, text):
        return super().embed_query(f"search_query: {text}")


def create_vector_store(documents, persist_directory="sample_table_db/chroma_db"):
    """Create and persist ChromaDB vector store, or load it if already created"""

    embedding_model = NomicEmbeddings(
        model_name="nomic-ai/nomic-embed-text-v1.5",
        model_kwargs={"trust_remote_code": True}
    )

    db_file = os.path.join(persist_directory, "chroma.sqlite3")

    if os.path.exists(db_file):
        print(f"✅ Vector store already created at {persist_directory} — loading existing DB")
        vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=embedding_model,
            collection_metadata={"hnsw:space": "cosine"}
        )
    else:
        print("🔮 Creating embeddings and storing in ChromaDB...")
        print("--- Creating vector store ---")
        vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=embedding_model,
            persist_directory=persist_directory,
            collection_metadata={"hnsw:space": "cosine"},
            ids=[f"chunk_{i}" for i in range(len(documents))]
        )
        print("--- Finished creating vector store ---")
        print(f"✅ Vector store created and saved to {persist_directory}")

    return vectorstore

db = create_vector_store(processed_chunks)

<All keys matched successfully>


🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to sample_table_db/chroma_db


In [37]:
def generate_final_answer(chunks, query):
    """Generate final answer + structured source list"""
    sources = []
    has_images = False

    try:
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        for i, chunk in enumerate(chunks):
            meta = chunk.metadata
            prompt_text += f"--- Document {i+1} (page {meta.get('page')}) ---\n"
            prompt_text += f"TEXT:\n{meta.get('raw_text', chunk.page_content)}\n\n"

            table_paths = json.loads(meta.get("table_paths", "[]"))
            table_image_paths = json.loads(meta.get("table_image_paths", "[]"))
            image_paths = json.loads(meta.get("image_paths", "[]"))
            
            if table_paths:
                source_entry["type"] = "table"
                source_entry["paths"].extend(table_image_paths if table_image_paths else table_paths)

            source_entry = {
                "chunk_id": meta.get("chunk_id"),
                "page": meta.get("page"),
                "type": "text",
                "preview": chunk.page_content[:150],
                "paths": []
            }
            if table_paths:
                source_entry["type"] = "table"
                source_entry["paths"].extend(table_paths)
            if image_paths:
                has_images = True
                source_entry["type"] = "image" if not table_paths else "table+image"
                source_entry["paths"].extend(image_paths)

            sources.append(source_entry)
            prompt_text += "\n"

        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        model_name = "qwen/qwen3.6-27b" if has_images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        if has_images:
            message_content = [{"type": "text", "text": prompt_text}]
            for chunk in chunks:
                image_paths = json.loads(chunk.metadata.get("image_paths", "[]"))
                for path in image_paths:
                    with open(path, 'rb') as f:
                        img_b64 = base64.b64encode(f.read()).decode('utf-8')
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_b64}"}
                    })
        else:
            message_content = prompt_text

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content, sources

    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer.", sources

In [38]:
def render_table_html(raw_html):
    """Wrap unstructured's table HTML with proper styling"""
    return f"""
    <style>
        .rag-table-wrapper {{
            font-family: -apple-system, sans-serif;
            font-size: 13px;
            overflow-x: auto;
            margin: 10px 0;
        }}
        .rag-table-wrapper table {{
            border-collapse: collapse;
            width: 100%;
        }}
        .rag-table-wrapper th, .rag-table-wrapper td {{
            border: 1px solid #ccc;
            padding: 6px 10px;
            text-align: left;
        }}
        .rag-table-wrapper th {{
            background-color: #f0f0f0;
            font-weight: 600;
        }}
        .rag-table-wrapper tr:nth-child(even) {{
            background-color: #fafafa;
        }}
    </style>
    <div class="rag-table-wrapper">{raw_html}</div>
    """


def display_query_result(query, answer, sources):
    print(f"❓ Query: {query}\n")
    print(f"💡 Answer:\n{answer}\n")
    print(f"📚 Sources ({len(sources)}):\n")

    output_area = widgets.Output()

    def make_click_handler(source):
        def handler(b):
            with output_area:
                clear_output(wait=True)
                print(f"--- chunk {source['chunk_id']} | page {source['page']} | type: {source['type']} ---\n")
                if not source['paths']:
                    print(source['preview'])
                for path in source['paths']:
                    if path.endswith(('.png', '.jpg', '.jpeg')):
                        display(IPImage(filename=path))
                    elif path.endswith('.html'):
                        with open(path, 'r', encoding='utf-8') as f:
                            raw_html = f.read()
                        display(HTML(render_table_html(raw_html)))   # ← the actual fix
        return handler

    buttons = []
    for i, source in enumerate(sources):
        label = f"[{i+1}] page {source['page']} • {source['type']}"
        btn = widgets.Button(description=label, layout=widgets.Layout(width='auto'))
        btn.on_click(make_click_handler(source))
        buttons.append(btn)

    display(widgets.HBox(buttons))
    display(output_area)

In [39]:
import re

def smart_retrieve(query, db, processed_chunks, k=5):
    """Retrieve normally, but force-include table chunks if query mentions a table"""
    results = db.similarity_search(query, k=k)

    if re.search(r'\btable\s*\d*\b', query, re.IGNORECASE):
        table_docs = [d for d in processed_chunks if d.metadata.get("has_table")]
        seen_ids = {d.metadata.get("chunk_id") for d in results}
        for doc in table_docs:
            if doc.metadata.get("chunk_id") not in seen_ids:
                results.append(doc)
                seen_ids.add(doc.metadata.get("chunk_id"))

    return results

In [ ]:
query = "What are the column headers and data cells in Table 1?"

retrieved_chunks = smart_retrieve(query, db, processed_chunks, k=5)   # ← changed from retriever.invoke(query)

answer, sources = generate_final_answer(retrieved_chunks, query)
display_query_result(query, answer, sources)

❓ Query: What are the column headers and data cells in Table 1?

💡 Answer:
I don’t have enough information to answer that question based on the provided documents.

📚 Sources (32):



Output()

In [9]:
import os
import camelot

def extract_tables_camelot(pdf_path, min_accuracy=70, verbose=True):
    """
    Extract tables from a PDF using Camelot 2.0, trying multiple flavors.
    Order: stream (whitespace/borderless) -> lattice (ruled) -> ml (neural, if installed)
    """
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return []

    print(f"📄 Extracting tables from: {pdf_path}")
    all_tables = []

    # --- Try stream first: best for whitespace-aligned / borderless tables ---
    try:
        stream_tables = camelot.read_pdf(pdf_path, pages="all", flavor="stream")
        if verbose:
            print(f"   [stream] found {len(stream_tables)} table(s)")
        for t in stream_tables:
            acc = t.parsing_report.get("accuracy", 0)
            if verbose:
                print(f"      page {t.page} | accuracy {acc:.1f}% | shape {t.df.shape}")
            if acc >= min_accuracy:
                all_tables.append({
                    "page": t.page,
                    "dataframe": t.df,
                    "html": t.df.to_html(index=False, header=False),
                    "accuracy": acc,
                    "method": "stream"
                })
    except Exception as e:
        print(f"   ⚠️ stream mode failed: {e}")

    # --- Try lattice for pages stream may have missed (ruled/bordered tables) ---
    pages_found = {t["page"] for t in all_tables}
    try:
        lattice_tables = camelot.read_pdf(pdf_path, pages="all", flavor="lattice")
        if verbose:
            print(f"   [lattice] found {len(lattice_tables)} table(s)")
        for t in lattice_tables:
            if t.page in pages_found:
                continue  # already caught by stream
            acc = t.parsing_report.get("accuracy", 0)
            if verbose:
                print(f"      page {t.page} | accuracy {acc:.1f}% | shape {t.df.shape}")
            if acc >= min_accuracy:
                all_tables.append({
                    "page": t.page,
                    "dataframe": t.df,
                    "html": t.df.to_html(index=False, header=False),
                    "accuracy": acc,
                    "method": "lattice"
                })
                pages_found.add(t.page)
    except Exception as e:
        print(f"   ⚠️ lattice mode failed: {e}")

    # --- Optional: neural backend for anything still missed (requires camelot-py[ml]) ---
    if not all_tables:
        try:
            ml_tables = camelot.read_pdf(pdf_path, pages="all", flavor="ml")
            if verbose:
                print(f"   [ml] found {len(ml_tables)} table(s)")
            for t in ml_tables:
                acc = t.parsing_report.get("accuracy", 0)
                all_tables.append({
                    "page": t.page,
                    "dataframe": t.df,
                    "html": t.df.to_html(index=False, header=False),
                    "accuracy": acc,
                    "method": "ml"
                })
        except ImportError:
            print("   ℹ️  'ml' flavor not available — install with: pip install \"camelot-py[ml]\"")
        except Exception as e:
            print(f"   ⚠️ ml mode failed: {e}")

    print(f"\n✅ Total tables extracted: {len(all_tables)}")
    for t in all_tables:
        print(f"   page {t['page']} | {t['method']} | accuracy {t['accuracy']:.1f}% | shape {t['dataframe'].shape}")

    return all_tables


# --- Run it ---
pdf_path = "./docs/attention-is-all-you-need.pdf"  # ← confirm this is the correct file before running
tables = extract_tables_camelot(pdf_path)

# Inspect a specific table
if tables:
    print("\n--- First table preview ---")
    print(tables[0]["dataframe"])

📄 Extracting tables from: ./docs/attention-is-all-you-need.pdf


c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (98.0, 94.5072576, 514.3140640576001, 234.60110706666666)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
c:\Users\chait\Documents\smk\Personal_Projects\Multi_Model_RAG\.venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (98.0, 659.6790784, 513.9972105879998, 768.325786877612)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


   [stream] found 17 table(s)
      page 1 | accuracy 100.0% | shape (16, 1)
      page 1 | accuracy 100.0% | shape (12, 1)
      page 2 | accuracy 73.4% | shape (22, 2)
      page 3 | accuracy 100.0% | shape (22, 1)
      page 4 | accuracy 100.0% | shape (13, 1)
      page 5 | accuracy 100.0% | shape (4, 1)
      page 5 | accuracy 94.5% | shape (16, 2)
      page 6 | accuracy 91.2% | shape (9, 4)
      page 7 | accuracy 100.0% | shape (18, 1)
      page 8 | accuracy 92.9% | shape (15, 5)
      page 9 | accuracy 98.0% | shape (29, 7)
      page 10 | accuracy 98.7% | shape (14, 3)
      page 11 | accuracy 100.0% | shape (44, 1)
      page 12 | accuracy 100.0% | shape (31, 1)
      page 13 | accuracy 100.0% | shape (33, 2)
      page 14 | accuracy 100.0% | shape (43, 4)
      page 15 | accuracy 100.0% | shape (27, 4)
   [lattice] found 2 table(s)

✅ Total tables extracted: 17
   page 1 | stream | accuracy 100.0% | shape (16, 1)
   page 1 | stream | accuracy 100.0% | shape (12, 1)
   page

In [7]:
print(f"Total tables found: {len(tables)}")
for t in tables:
    acc = t.parsing_report.get("accuracy", 0)
    print(f"Page {t.page} | accuracy {acc:.1f}% | shape {t.df.shape}")

Total tables found: 0
